# Machine Learning Pipeline

Now that we have experience preparing data for input to machine learning libraries, the next step will be to train, tune, and test a model.  You will perform all three of these steps in this hands-on activity.

The assignment consists of the following steps:

1. Load two datasets and prepare their representations and labels for model input. 
2. Split the data into training and testing.
3. Select a model, and identify the parameters to tune.
4. Tune the model.
5. Evaluate the model's performance.

In [25]:
import logging
logging.getLogger("scapy.runtime").setLevel(logging.ERROR)

from netml.pparser.parser import PCAP
from netml.utils.tool import dump_data, load_data

import pandas as pd
from sklearn.model_selection import train_test_split

ImportError: DLL load failed while importing _fblas: An Application Control policy has blocked this file.

## Convert the Packet Capture Into Flows

1. Load the two packet captures for HTTP requests and Log4j scan, 
2. convert them into traffic flows, 
3. generate features from the flow,  
4. label the traffic,
5. normalize your labeled features into a 2D matrix

In [3]:
http = PCAP("data/http.pcap", 
            flow_ptks_thres=2,
            random_state=42,
            verbose=10)
logj = PCAP("data/log4j.pcap",
            flow_ptks_thres=2,
            random_state=42,
            verbose=10)

In [4]:
http.pcap2flows(q_interval=0.9)
logj.pcap2flows(q_interval=0.9)

pcap_file: data/http.pcap
ith_packets: 0
ith_packets: 10000
ith_packets: 20000
len(flows): 593
total number of flows: 593. Num of flows < 2 pkts: 300, and >=2 pkts: 293 without timeout splitting.
kept flows: 293. Each of them has at least 2 pkts after timeout splitting.
flow_durations.shape: (293, 1)
        col_0
count 293.000
mean   11.629
std    15.820
min     0.000
25%     0.076
50%     0.455
75%    20.097
max    46.235
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293 entries, 0 to 292
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   col_0   293 non-null    float64
dtypes: float64(1)
memory usage: 2.4 KB
None
0th_flow: len(pkts): 4
After splitting flows, the number of subflows: 291 and each of them has at least 2 packets.
pcap_file: data/log4j.pcap
ith_packets: 0
ith_packets: 10000
ith_packets: 20000
ith_packets: 30000
ith_packets: 40000
ith_packets: 50000
ith_packets: 60000
ith_packets: 70000
ith_packets: 80000
len

In [5]:
http.flow2features('IAT')
http_iat = http.features #need to label features

True


In [6]:
logj.flow2features('IAT')
logj_iat = logj.features #features for log4j, label as log4j

True


In [7]:
http.label_flows(label=0)
logj.label_flows(label=1)

In [14]:
http_df = pd.DataFrame(http.features)
http_df['label'] = http.labels

In [23]:
features = http_df.iloc[:-1].to_numpy()
features

array([[3.69608402e-02, 3.00027771e+01, 3.66208553e-02, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.95369720e-02, 3.00200751e+01, 1.99518204e-02, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.54939148e+01, 4.99856710e+00, 1.51307790e+01, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [1.32579803e-02, 3.00407410e-05, 5.41925430e-04, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.85840130e-02, 1.95097923e-03, 1.91428661e-02, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [2.44238377e-02, 8.59975815e-04, 7.91549683e-05, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]])

In [15]:
logj_df = pd.DataFrame(logj.features)
logj_df['label'] = logj.labels

## Evaluating a Machine Learning Model

The goal of supervised learning is to train a model that takes examples and predicts labels for these examples that are as close as possible to the actual labels. For instance, in this example above, a model might take features from a traffic trace and predict whether the traffic constitutes regular web traffic or a scan.

How do you measure whether the model is succeeding if you don't know the true labels for new observations? The way to solve this problem is to test the performance of the trained algorithm on additional data that it has never seen, but for which you already know the correct labels. 

This requires that you train the algorithm using only a portion of the entire labeled dataset (the **training set**) and withold the rest of the labeled data (the **test set**) for testing how well the model generalizes to new information. 

To evaluate the model, we will need to split the data into train and test sets.

### Split into Training and Test Sets

Split your data into a training and test set using scikit-learn. A common split is to train on 80% of your data, while withholding 20% of the data. 

### Training Your Model

Now that you have split your data into training and testing sets, you are ready to train and evaluate a model. 

Import a machine learning model of your choice, use your training set to train the model, and use the test set to evaluate it. 

### Test Your Trained Model

You can now evaluate how well your trained model works.  There are several valuable ways to visualize your results. You might use techniques such as a confusion matrix, or a receiver operating characteristic (ROC) curve. Below we will gain some experience plotting both of those.  This [documentation](https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_display_object_visualization.html) may help you with plotting these results.

#### Confusion Matrix 

A confusion matrix is a one way to understand errors of different types. We can see a lot of examples off diagonal, suggesting a fair number of incorrect answers.

#### Receiver Operating Characteristic

Some models can output different classes based on a threshold that is set for the decision. 

#### Area Under the Curve (AUC)

From the ROC above, you can also compute a metric called the area under the curve (AUC). Visually, this is the area under the curve that you just plotted. You could see, intuitively, that the "best" performance should yield an AUC of 1, and the worst performance would yield an AUC closer to 0.5.

Scikit learn also has a function for computing AUC.  Compute the area under the curve.

## Thought Question

Which evaluation model is more appropriate, and when (i.e., under what circumstances)? When might you care more about looking at the confusion matrix (or model accuracy) vs. the ROC, or the area under the curve?